## What are Medallion Architecture? (Bronze, Silver, Gold)

先明白奖章架构Medallion Architecture

**每一层（Bronze、Silver、Gold）都代表了数据在生命周期中不同的“法定完工状态”。明确划分各层的职责，是为了在“原始数据的绝对真实性”与“业务层面的极端易用性”之间，拉出三道无法逾越的物理防火墙。**

**如果职责划分不清，就会导致数仓退化为污水直排的垃圾场。**

在工业界，这三层的标准职责划分以及标准的湖仓数据流转流程如下：

#### 1. Bronze 贴源层 

Bronze 层的核心天职是 **Immutable Data Ingestion（不变量数据落地）**。

* **物理职责**：原封不动地存储所有从上游业务系统（MySQL Binlog、前端埋点日志、Kafka 流）同步过来的原始数据。
* **重工业红线**：
* **零逻辑清洗**：不允许过滤任何看似“没用”的行，不允许修改任何脏数据类型。必须保持原汁原味的最初状态。
* **纯追加模式（Append-Only）**：来什么存什么，用时间戳记录历史的每一次漂移。
* **长周期存储**：作为数仓最后的物理退路，一旦下游模型全部改版，能够依靠 Bronze 层的数据重新从头跑出整座数仓。



#### 2. Silver 清洗与规范层

Silver 层的核心天职是 **Cleansing, Conforming & Enrichment（清洗、对齐与富化）**。

* **物理职责**：把 Bronze 层的无序数据转化为全公司统一、干净、可直接用于建模的“标准件”。
* **重工业红线**：
* **精准去重（Deduplication）**：利用 `dropDuplicates` 或窗口函数，斩断分布式传输中不可避免产生的物理重复流。
* **结构约束（Schema Enforcement）**：强行进行类型转换（如把字符串 `12999.0` 转换为 `Decimal(10,2)`），过滤非法乱码，缺省值补 `null`。
* **维度富化（Enrichment）**：在此层完成基础维度的拓宽，例如通过 `user_id` 自动关联出当时该用户的 `country_code` 或 `age_group`。



#### 3. Gold 业务与聚合层 

Gold 层的核心天职是 **Business-Level Analytics & Modeling（业务级分析与建模）**。

* **物理职责**：针对特定的业务线（如财务、风控、高频量化、运营看板），组装出高度压缩、开箱即用的宽表、指标集或数据立方体（Data Cube）。
* **重工业红线**：
* **降维大聚合**：将明细级的流水数据，按照特定维度（如按天、按国家、按卖家）进行高效的聚合统计。
* **手术刀级增量（Upsert/Merge）**：全线采用 `MERGE INTO` 等原子事务算子，确保最新状态无痛合拢。
* **性能极致优化**：在此层强行执行 `OPTIMIZE`、加盐以及合理的 `Z-ORDER` 物理文件级排序，确保下游报表点击时能够实现毫秒级“分区裁剪”响应。



---

### 标准湖仓流程 The Standard Lakehouse Data Flow

在重工业生产中，一条标准的数据从诞生到消费，需要流经以下 6 步闭环流程：

```text
 ┌─────────────┐     ┌─────────────┐     ┌─────────────┐     ┌─────────────┐
 │  Data Source│ ──► │ Bronze Layer│ ──► │ Silver Layer│ ──► │ Gold Layer  │
 └─────────────┘     └─────────────┘     └─────────────┘     └─────────────┘
   (Kafka/CDC)        (Append-Only)       (Deduplicate)       (Merge/Upsert)
                                                                    │
                                                                    ▼
 ┌─────────────┐     ┌─────────────┐                         ┌─────────────┐
 │ Data Users  │ ◄── │ Data Audit  │ ◄───────────────────────│  Storage OS │
 └─────────────┘     └─────────────┘                         └─────────────┘
  (BI/AI Apps)      (Financial Assert)                       (OPTIMIZE/VACUUM)

```

1. **增量捕获（Ingestion / CDC）**：通过 Kafka（实时日志）或 Debezium（MySQL 增量同步），将最新批次或流式数据不间断地倾倒进 **Bronze** 表。
2. **防腐清洗（Cleansing）**：Silver 任务被唤醒，订阅 Bronze 的最新版本，拉网式执行字段约束、物理去重、乱码斩断，然后将干净的数据写进 **Silver** 表。
3. **高维合拢（Aggregation）**：Gold 任务被唤醒，订阅 Silver 清洗后的干净实体，通过主键对撞，利用 `MERGE INTO` 增量合拢历史资产线。
4. **后置文件治理（Storage Maintenance）**：在 Gold 层落盘成功后，自动异步触发 `OPTIMIZE` 合并小文件碎块，并定期执行 `VACUUM` 清理过期历史 Parquet 碎片，牢牢锁死物理存储成本。
5. **金融级资产对账（Data Audit & Quality）**：触发全链路强断言控制线。比对 `Bronze.Rows == Silver.Rows + Duplicates` 以及财务总额字段的 `SUM()` 平衡。
6. **资产安全上架（Consumption）**：通过统一的元数据中心（Catalog）对下游分析师、BI 报表、大模型特征工程安全上架，提供零死锁、快照隔离的高性能全量只读查询。
